In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

In [0]:
dataset = spark.table(
    "agentdb.forecasting.forecast_training_dataset"
)

baseline_predictions = (

    dataset

    .select(
        "product_key",
        "store_key",
        "calendar_key",

        "target_sales_qty",

        col(
            "rolling_7d_avg"
        ).alias(
            "predicted_demand"
        )
    )
)

baseline_output = (

    baseline_predictions

    .withColumn(
        "model_name",
        lit("BASELINE_7D_AVG")
    )

    .withColumn(
        "forecast_horizon_days",
        lit(1)
    )

    .withColumn(
        "created_at",
        current_timestamp()
    )
    
    .withColumn(
        "prediction_id",
        monotonically_increasing_id()
    )
    
    .withColumn(
        "forecast_date",
        current_date()
    )
)

(
    baseline_output
    .select(
        "prediction_id",
        "product_key",
        "store_key",
        "forecast_date",
        "forecast_horizon_days",
        "predicted_demand",
        "model_name",
        "created_at"
    )
    .write
    .mode("overwrite")
    .saveAsTable(
        "agentdb.forecasting.forecast_predictions"
    )
)

In [0]:
weighted_predictions = (

    dataset

    .withColumn(

        "predicted_demand",

        (
            col("rolling_7d_avg") * 0.7
        )
        +
        (
            col("rolling_30d_avg") * 0.3
        )
    )
)
weighted_output = (

    weighted_predictions

    .withColumn(
        "model_name",
        lit(
            "WEIGHTED_7D_30D"
        )
    )

    .withColumn(
        "forecast_horizon_days",
        lit(1)
    )

    .withColumn(
        "created_at",
        current_timestamp()
    )
    
    .withColumn(
        "prediction_id",
        monotonically_increasing_id()
    )
    
    .withColumn(
        "forecast_date",
        current_date()
    )
)

(
    weighted_output
    .select(
        "prediction_id",
        "product_key",
        "store_key",
        "forecast_date",
        "forecast_horizon_days",
        "predicted_demand",
        "model_name",
        "created_at"
    )
    .write
    .mode("append")
    .saveAsTable(
        "agentdb.forecasting.forecast_predictions"
    )
)

In [0]:
predictions = spark.table(
    "agentdb.forecasting.forecast_predictions"
)

training_data = spark.table(
    "agentdb.forecasting.forecast_training_dataset"
)

predictions_with_actuals = (
    predictions
    .join(
        training_data.select(
            "product_key",
            "store_key",
            "calendar_key",
            "target_sales_qty"
        ),
        on=["product_key", "store_key"],
        how="inner"
    )
)

evaluation = (

    predictions_with_actuals

    .groupBy(
        "model_name"
    )

    .agg(

        avg(
            abs(
                col("target_sales_qty")
                -
                col("predicted_demand")
            )
        ).alias(
            "mae"
        ),

        sqrt(
            avg(
                pow(
                    col("target_sales_qty")
                    -
                    col("predicted_demand"),
                    2
                )
            )
        ).alias(
            "rmse"
        )
    )
)

evaluation = (
    evaluation
    .withColumn(
        "evaluation_date",
        current_date()
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
)

evaluation.write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.forecasting.forecast_metrics"
    )

In [0]:
%sql
SELECT * FROM agentdb.forecasting.forecast_predictions
LIMIT 10;

In [0]:
%sql
SELECT * FROM agentdb.forecasting.forecast_metrics
LIMIT 10;